# Perbandingan CNN From Scratch dan Transfer Learning

## Tujuan
Notebook ini melanjutkan tugas minggu 9 menggunakan dataset **Fashion-MNIST dari Kaggle**. Tiga pendekatan dibandingkan pada pembagian data yang sama:

1. CNN sederhana yang dilatih **from scratch**.
2. **MobileNetV2** pretrained ImageNet dengan classifier baru.
3. **EfficientNetB0** pretrained ImageNet dengan classifier baru.

Perbandingan dilakukan menggunakan accuracy, precision, recall, dan F1-score weighted pada test set. Dua model transfer learning memakai bobot ImageNet yang dibekukan sehingga eksperimen mengukur manfaat feature extractor yang sudah dipelajari.

In [1]:
# Jalankan sekali jika dependensi belum tersedia.
%pip install -q kagglehub tensorflow scikit-learn pandas matplotlib seaborn jinja2
# Setelah instalasi selesai, pilih Kernel > Restart Kernel, lalu gunakan Run All.
# Restart wajib dilakukan agar NumPy/TensorFlow dimuat dari state kernel yang bersih.

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\Ilham Febriyansyah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
EPOCHS = 5
BATCH_SIZE = 64
NUM_CLASSES = 10
INPUT_SIZE = (96, 96)
np.random.seed(SEED)
tf.random.set_seed(SEED)
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

C:\Users\Ilham Febriyansyah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TensorFlow: 2.21.0
GPU: []


## 1. Dataset Kaggle dan preprocessing

Fashion-MNIST terdiri dari 60.000 data train dan 10.000 data test, berukuran 28x28 grayscale dan memiliki 10 kelas. Data train dibagi menjadi train/validasi secara stratified. Test set tidak digunakan saat training.

In [3]:
# Cell ini dapat dijalankan setelah cell import, atau secara mandiri.
# Jika muncul 'CPU dispatcher tracer already initialized', restart kernel lalu Run All.
from pathlib import Path
import kagglehub
import pandas as pd

DATASET_DIR = Path(kagglehub.dataset_download("zalando-research/fashionmnist"))
csv_files = sorted(DATASET_DIR.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError("CSV Fashion-MNIST dari Kaggle tidak ditemukan.")

def read_split(name):
    candidates = [p for p in csv_files if name in p.stem.lower()]
    if not candidates:
        raise FileNotFoundError(f"Split {name!r} tidak ditemukan.")
    return pd.read_csv(candidates[0])

train_df = read_split("train")
test_df = read_split("test")
pixel_columns = [c for c in train_df.columns if c != "label"]
if len(pixel_columns) != 784:
    raise ValueError(f"Jumlah kolom piksel tidak sesuai: {len(pixel_columns)}")

X = train_df[pixel_columns].to_numpy(dtype="float32").reshape(-1, 28, 28, 1) / 255.0
y = train_df["label"].to_numpy(dtype="int64")
X_test = test_df[pixel_columns].to_numpy(dtype="float32").reshape(-1, 28, 28, 1) / 255.0
y_test = test_df["label"].to_numpy(dtype="int64")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}; validasi: {X_val.shape}; test: {X_test.shape}")

Train: (54000, 28, 28, 1); validasi: (6000, 28, 28, 1); test: (10000, 28, 28, 1)


In [4]:
def make_dataset(images, labels, batch_size=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

scratch_train = make_dataset(X_train, y_train, shuffle=True)
scratch_val = make_dataset(X_val, y_val)
scratch_test = make_dataset(X_test, y_test)

# Transfer learning memerlukan input RGB dan ukuran minimum yang lebih besar.
def to_rgb_96(images, labels):
    images = tf.image.grayscale_to_rgb(images)
    images = tf.image.resize(images, INPUT_SIZE)
    return images, labels

def make_transfer_dataset(images, labels, shuffle=False):
    return make_dataset(images, labels, shuffle=shuffle).map(
        to_rgb_96, num_parallel_calls=tf.data.AUTOTUNE
    ).prefetch(tf.data.AUTOTUNE)

transfer_train = make_transfer_dataset(X_train, y_train, shuffle=True)
transfer_val = make_transfer_dataset(X_val, y_val)
transfer_test = make_transfer_dataset(X_test, y_test)
print("Rentang piksel:", float(X_train.min()), "sampai", float(X_train.max()))

Rentang piksel: 0.0 sampai 1.0


## 2. Model from scratch

CNN ini mempelajari seluruh filter konvolusi dari bobot acak. Arsitekturnya sengaja dibuat sederhana sebagai baseline minggu 9.

In [5]:
def build_scratch_model():
    return tf.keras.Sequential([
        tf.keras.layers.Input((28, 28, 1)),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu"),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")
    ], name="CNN_From_Scratch")

scratch_model = build_scratch_model()
scratch_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
scratch_history = scratch_model.fit(
    scratch_train, validation_data=scratch_val, epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=1
)

Epoch 1/5


C:\Users\Ilham Febriyansyah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


844/844 ━━━━━━━━━━━━━━━━━━━━ 28s 28ms/step - accuracy: 0.7200 - loss: 0.7627 - val_accuracy: 0.8092 - val_loss: 0.5206
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 24s 28ms/step - accuracy: 0.8174 - loss: 0.5018 - val_accuracy: 0.8383 - val_loss: 0.4522
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 23s 27ms/step - accuracy: 0.8451 - loss: 0.4317 - val_accuracy: 0.8590 - val_loss: 0.3860
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 23s 27ms/step - accuracy: 0.8583 - loss: 0.3913 - val_accuracy: 0.8685 - val_loss: 0.3710
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 23s 27ms/step - accuracy: 0.8699 - loss: 0.3604 - val_accuracy: 0.8770 - val_loss: 0.3439


## 3. Dua metode transfer learning

Kedua backbone menggunakan bobot ImageNet (`weights="imagenet"`) dan `include_top=False`. Hanya global pooling, dropout, dan dense classifier yang dilatih pada Fashion-MNIST. Jika bobot belum tersimpan di komputer, TensorFlow akan mengunduhnya saat cell pertama kali dijalankan.

In [ ]:
def build_transfer_model(backbone_name):
    if backbone_name == "MobileNetV2":
        backbone = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", input_shape=(*INPUT_SIZE, 3)
        )
        preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
    elif backbone_name == "EfficientNetB0":
        backbone = tf.keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet", input_shape=(*INPUT_SIZE, 3)
        )
        preprocess = tf.keras.applications.efficientnet.preprocess_input
    else:
        raise ValueError(f"Backbone tidak didukung: {backbone_name}")

    backbone.trainable = False
    inputs = tf.keras.Input(shape=(*INPUT_SIZE, 3))
    x = preprocess(inputs * 255.0)
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name=backbone_name + "_TransferLearning")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

mobilenet_model = build_transfer_model("MobileNetV2")
mobilenet_history = mobilenet_model.fit(
    transfer_train, validation_data=transfer_val, epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=1
)

efficientnet_model = build_transfer_model("EfficientNetB0")
efficientnet_history = efficientnet_model.fit(
    transfer_train, validation_data=transfer_val, epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=1
)

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 249s 285ms/step - accuracy: 0.8274 - loss: 0.4893 - val_accuracy: 0.8817 - val_loss: 0.3391
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 236s 280ms/step - accuracy: 0.8715 - loss: 0.3554 - val_accuracy: 0.8827 - val_loss: 0.3229
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 237s 280ms/step - accuracy: 0.8777 - loss: 0.3333 - val_accuracy: 0.8905 - val_loss: 0.3040
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 237s 280ms/step - accuracy: 0.8815 - loss: 0.3248 - val_accuracy: 0.8858 - val_loss: 0.3099
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 236s 279ms/step - accuracy: 0.8842 - loss: 0.3207 - val_accuracy: 0.8843 - val_loss: 0.3084
Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 361s 408ms/step - accuracy: 0.8290 - loss: 0.4946 - val_accuracy: 0.8737 - val_loss: 0.3436
Epoch 2/5
 80/844 ━━━━━━━━━━━━━━━━━━━━ 5:13 410ms/step - accuracy: 0.8633 - loss: 0.3674

## 4. Evaluasi accuracy, precision, recall, dan F1-score

Precision, recall, dan F1-score menggunakan `average="weighted"`, sehingga kontribusi setiap kelas ditimbang berdasarkan jumlah sampelnya. Selain tabel, confusion matrix untuk model terbaik ditampilkan.

In [ ]:
def evaluate_model(model, dataset, name):
    probabilities = model.predict(dataset, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    labels = np.concatenate([batch_y.numpy() for _, batch_y in dataset])
    return {
        "Metode": name,
        "Accuracy": accuracy_score(labels, predictions),
        "Precision (weighted)": precision_score(labels, predictions, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(labels, predictions, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(labels, predictions, average="weighted", zero_division=0),
        "labels": labels,
        "predictions": predictions,
    }

results = [
    evaluate_model(scratch_model, scratch_test, "CNN from scratch"),
    evaluate_model(mobilenet_model, transfer_test, "MobileNetV2 transfer learning"),
    evaluate_model(efficientnet_model, transfer_test, "EfficientNetB0 transfer learning"),
]
results_df = pd.DataFrame([{k: v for k, v in result.items() if k not in {"labels", "predictions"}} for result in results])
display(results_df.round(4))
results_df.to_csv("hasil_perbandingan.csv", index=False)
print("Hasil disimpan ke hasil_perbandingan.csv")

NameError: name 'scratch_model' is not defined

In [ ]:
best_index = int(results_df["F1-score (weighted)"].to_numpy().argmax())
best_result = results[best_index]
print("Classification report:", results_df.loc[best_index, "Metode"])
print(classification_report(best_result["labels"], best_result["predictions"], target_names=CLASS_NAMES, zero_division=0))

plt.figure(figsize=(9, 7))
sns.heatmap(
    confusion_matrix(best_result["labels"], best_result["predictions"]),
    annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.title("Confusion matrix - " + results_df.loc[best_index, "Metode"])
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Report perbandingan

| Aspek | From scratch | Transfer learning |
|---|---|---|
| Inisialisasi | Bobot acak | Bobot ImageNet yang sudah dipelajari |
| Feature extractor | Dipelajari dari nol pada Fashion-MNIST | Backbone dibekukan, classifier baru dilatih |
| Input | 28x28 grayscale | Diubah menjadi RGB 96x96 |
| Kebutuhan data dan waktu | Biasanya membutuhkan lebih banyak epoch/data untuk fitur yang baik | Konvergensi awal biasanya lebih cepat karena fitur umum sudah tersedia |
| Risiko | Lebih mudah underfit pada epoch sedikit | Domain ImageNet berbeda dari Fashion-MNIST; resize dan grayscale dapat membatasi manfaat |

Kesimpulan akhir harus diambil dari `results_df` setelah semua cell selesai dijalankan. Model terbaik ditentukan oleh F1-score weighted, bukan hanya accuracy. Nilai evaluasi di atas adalah hasil eksperimen aktual pada run notebook ini dan dapat berubah sedikit jika seed, jumlah epoch, versi TensorFlow, atau perangkat keras berbeda.